# learn-pytorch — run on Colab GPU

Workflow: edit code locally → `git push` → re-run the pull cell below → run whichever script cell you want.

Runtime → Change runtime type → select **T4 GPU** (or better) before running.

In [ ]:
REPO_URL = "https://github.com/borisepshtein/learn-pytorch.git"
REPO_DIR = "/content/learn-pytorch"

import os
if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Push results back to GitHub (closes the feedback loop)

Each script saves its output image + a small metrics JSON (latest run only) under `results/`, and also appends a timestamped record to `results/experiment_log.jsonl` so metrics from every past run stay comparable across code changes. This cell commits and pushes that folder back to GitHub, so results can be pulled and read directly outside Colab without any manual copy-paste.

In [ ]:
import datetime

def push_results(label):
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        print('GITHUB_TOKEN secret not found. Add it via the key icon (Secrets) in the left sidebar: '
              'a GitHub fine-grained PAT scoped to this repo with Contents: Read and write permission.')
        raise
    push_url = f'https://{token}@github.com/borisepshtein/learn-pytorch.git'
    ts = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')
    !git config user.email "boris.epshtein@gmail.com"
    !git config user.name "Boris Epshtein"
    !mkdir -p results
    !git add -A results
    !git commit -m "Colab run results: {label} ({ts})" || echo "Nothing new to commit"
    !git push {push_url} main

## Script: MNIST binary classifier (digit 3 vs. rest) — *disabled, see cells below*

In [ ]:
# !python mnist_classifier.py

In [ ]:
# push_results("mnist_classifier")

## Script: denoising/deblurring/dewarping autoencoder + anomaly detection — *disabled, see cells below*

Trains only on digit 3 (blur + noise + elastic warp → restore original), then checks how well it restores unseen digits vs. digit 3s. Produces `results/autoencoder_anomaly.png` + `results/autoencoder_anomaly_metrics.json`.

In [ ]:
# !python mnist_autoencoder_anomaly.py

In [ ]:
# from IPython.display import Image, display
# display(Image(filename='results/autoencoder_anomaly.png'))

In [ ]:
# push_results("mnist_autoencoder_anomaly")

## Script: UCSD Ped2 video anomaly detection

Downloads the UCSD Anomaly Dataset (first run only, ~a few hundred MB, cached under `./data`), trains a conv autoencoder on normal pedestrian-walkway frames only, then scores every test frame's reconstruction error and computes a frame-level ROC/AUC against the dataset's ground-truth anomaly masks. Produces `results/ucsd_ped2_anomaly.png` + `results/ucsd_ped2_anomaly_metrics.json`.

In [ ]:
!python ucsd_ped2_autoencoder.py

In [ ]:
from IPython.display import Image, display
display(Image(filename='results/ucsd_ped2_anomaly.png'))

In [ ]:
push_results("ucsd_ped2_autoencoder")

## Script: SCUT-FBP5500 facial beauty classifier (ResNet18, Caucasian female subset)

Downloads the SCUT-FBP5500 dataset (Google Drive, first run only, ~172MB, cached under `./data`; non-commercial research use only), then runs 5-fold stratified cross-validation: for each fold, fine-tunes a fresh pretrained ResNet18 to classify Caucasian-female faces as pretty vs. average (median split on the dataset's mean human beauty rating), reporting per-fold accuracy/ROC-AUC plus the mean +/- std across folds (checks whether a single-split result is stable or a fluke of that split). The classifier's sigmoid confidence doubles as a per-face beauty score, and Grad-CAM overlays on fold 1's predictions show which facial regions drove each prediction. Produces `results/scut_fbp_beauty.png` + `results/scut_fbp_beauty_metrics.json`.

In [ ]:
!python scut_fbp_beauty_classifier.py

In [ ]:
from IPython.display import Image, display
display(Image(filename='results/scut_fbp_beauty.png'))

In [ ]:
push_results("scut_fbp_beauty_classifier")

## Script: SCUT-FBP5500 cross-race/gender beauty transfer test

Trains a ResNet18 on the Caucasian-female pretty/average split, then evaluates it both in-domain (held-out CF faces) and cross-domain (the full Asian-female subset, labeled by its own median split) -- then repeats in the opposite direction (train on AF, test on CF). Tests whether the beauty judgments learned from one population transfer to another, or collapse toward chance. Produces `results/scut_fbp_beauty_cross_race_transfer.png` + `results/scut_fbp_beauty_cross_race_transfer_metrics.json`.

In [ ]:
!python scut_fbp_beauty_cross_race_transfer.py

In [ ]:
from IPython.display import Image, display
display(Image(filename='results/scut_fbp_beauty_cross_race_transfer.png'))

In [ ]:
push_results("scut_fbp_beauty_cross_race_transfer")

## Script: SCUT-FBP5500 cross-race transfer, hair excluded (landmark face crop)

Follow-up on the cross-race transfer test: crops every face to its facial-landmark bounding box (eyebrows to chin, cheek to cheek -- excludes most hair) using the dataset's bundled 86-point landmarks, then reruns the same CF<->AF transfer test on those crops. Tests whether hair color/texture was part of why CF->AF transfer collapsed in calibration. Produces `results/scut_fbp_beauty_cross_race_transfer_no_hair.png` + `results/scut_fbp_beauty_cross_race_transfer_no_hair_metrics.json`.

In [ ]:
!python scut_fbp_beauty_cross_race_transfer_no_hair.py

In [ ]:
from IPython.display import Image, display
display(Image(filename='results/scut_fbp_beauty_cross_race_transfer_no_hair.png'))

In [ ]:
push_results("scut_fbp_beauty_cross_race_transfer_no_hair")

## Script: SCUT-FBP5500 age-controlled ablation

Estimates age for every CF (and AF, for comparison) image with an off-the-shelf age model (DeepFace, auto-installed), then builds an age-matched subset -- equal pretty/average counts within each 5-year age bin, so the two label groups have near-identical age distributions -- and retrains the baseline ResNet18 recipe on it. Also trains a same-size random (not age-matched) subsample as a control, so any accuracy drop can be attributed to age-matching specifically rather than just less data. Age estimates are cached to `./data` so re-runs skip the slow estimation step. Produces `results/scut_fbp_beauty_age_control.png` + `results/scut_fbp_beauty_age_control_metrics.json`.

In [ ]:
!python scut_fbp_beauty_age_control.py

In [ ]:
from IPython.display import Image, display
display(Image(filename='results/scut_fbp_beauty_age_control.png'))

In [ ]:
push_results("scut_fbp_beauty_age_control")

## Script: multi-source beauty classifier (SCUT + CFD + London), face-only crops

The user's actual goal, not just an audit: a classifier that doesn't pick up the wrong cues. Combines three independently-rated datasets with DIFFERENT rater pools (unlike the SCUT-only cross-race experiments, where all raters were the same 60 Asian people) -- SCUT-FBP5500, the Chicago Face Database (CFD), and the Face Research Lab London Set -- all cropped to face-only (excludes hair, confirmed earlier to be part of the shortcut). Runs leave-one-source-out cross-validation (train on 2 sources, test in-domain + out-of-domain on the 3rd) for all 3 combinations, then trains one final model on everything combined.

**CFD requires one-time setup**: it's gated behind a personal access request (https://www.chicagofaces.org/download/, not redistributable) -- upload the resulting zip to your Google Drive at `MyDrive/beauty_classifier_data/CFD.zip`. Run the Drive-mount cell immediately below (once per Colab session, prompts an authorization popup) *before* the script cell -- `drive.mount()` only works from an actual notebook cell, not from a script run via `!python`, so the script can't mount it itself. SCUT and the London Set download automatically, no setup needed; if CFD/Drive isn't found this just runs with 2 sources instead of 3.

Produces `results/beauty_classifier_multisource.png` + `results/beauty_classifier_multisource_metrics.json`.

In [ ]:
# Required once per Colab session before the next cell, so it can read CFD.zip from your Drive.
# drive.mount() only works from an actual notebook cell (needs the interactive kernel connection),
# not from a script run via !python, so this can't live inside beauty_classifier_multisource.py.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python beauty_classifier_multisource.py

In [ ]:
from IPython.display import Image, display
display(Image(filename='results/beauty_classifier_multisource.png'))

In [ ]:
push_results("beauty_classifier_multisource")